# elec_voids_search: полный геометрический скрининг

Порядок: **сетка + Вороной → общий список → близкие дубли → симметрийные группы → геометрия пробного иона**.
Сохраняются количество и координаты кандидатов каждого этапа, причины отказов и неопределённые случаи.

Радиусы каркаса находятся отдельно для каждого сайта. Пробный ион вставляется в копию исходной периодической ячейки, его КЧ и радиус Shannon определяются отдельно для каждой полости. Элемент и заряд задаются на входе. Поправка по наблюдаемым контактам не применяется. Это предварительный скрининг, не доказательство энергетической устойчивости. Анализ: [GEOMETRY_ANALYSIS.md](GEOMETRY_ANALYSIS.md).

Ноутбук подготовлен без выполнения. Ядро: `Desktop/electrolyties/elec_venv`. Все параметры: [PARAMETERS.md](PARAMETERS.md).

Пробник добавляется без расширения ячейки и без изменения дробных координат. Его периодические копии могут изменить КЧ относительно прежнего расчёта с расширением; сопоставимость результатов ещё не проверена.


In [1]:
from pathlib import Path
from datetime import datetime
from zoneinfo import ZoneInfo
from dataclasses import replace
from contextlib import redirect_stdout, redirect_stderr
from time import perf_counter
import ast
import hashlib
import json
import sys

import numpy as np
import pandas as pd
from IPython.display import display
from pymatgen.core import Structure
from pymatgen.symmetry.analyzer import SpacegroupAnalyzer

PROJECT_ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents]
                    if (p / "elec_voids_search/run_search.py").is_file())
sys.path.insert(0, str(PROJECT_ROOT))
from elec_voids_search import DEFAULT_CONFIG, SearchConfig
from elec_voids_search.run_search import run_search

# Дефолты берутся из config.py. Любое поле можно изменить через replace.
SETTINGS = SearchConfig(probe_element="Li", probe_oxidation_state=1)
# Примеры (раскомментируйте нужный):
# SETTINGS = replace(SETTINGS, grid_resolution_A=0.10)
# SETTINGS = replace(SETTINGS, merge_distance_A=0.15, merge_representative="first")
# SETTINGS = replace(SETTINGS, voronoi_image_range=2, symmetry_tolerance_A=0.05)
# SETTINGS = replace(SETTINGS, merge_distance_A=None)  # без общего объединения

# Другой ион (пример, не запускается автоматически):
# SETTINGS = replace(SETTINGS, probe_element="Na", probe_oxidation_state=1)
# SETTINGS = replace(SETTINGS, geometry_probe_spin="High Spin")
# SETTINGS = replace(SETTINGS, geometry_refine=False)
# Явные заряды относятся к каркасу; заряд внедряемого иона задан отдельно:
# SETTINGS = replace(SETTINGS, geometry_oxidation_states={"Li": 1, "Cl": -1})
# Как в старом скрипте, разрешить явно помеченный fallback к ionic_radius:
# SETTINGS = replace(SETTINGS, geometry_allow_ionic_fallback=True)
# Смягчить проверку до 0.75 радиуса (это уже не полный размер иона):
# SETTINGS = replace(SETTINGS, geometry_minimum_fit_fraction=0.75)

STAGES = ["grid", "voronoi", "combined", "merged", "nonequivalent",
          "geometry_evaluated", "geometry", "geometry_indeterminate",
          "geometry_rejected", "geometry_retained"]
VACANCY_TOLERANCE_A = 0.25  # критерий оценки известной позиции, не фильтр поиска
parameter_table = pd.DataFrame([
    {"parameter": key, "value": repr(value)} for key, value in SETTINGS.to_dict().items()
])
display(parameter_table)

run_id = datetime.now(ZoneInfo("Europe/Moscow")).strftime("%Y%m%d_%H%M%S_%f")
OUTPUT_DIR = PROJECT_ROOT / "elec_voids_results" / f"notebook_tests_{run_id}"
OUTPUT_DIR.mkdir(parents=True, exist_ok=False)
LOG_DIR = OUTPUT_DIR / "logs"
LOG_DIR.mkdir()
(OUTPUT_DIR / "settings.json").write_text(json.dumps(SETTINGS.to_dict(), indent=2))
print("Python:", sys.executable)
print("Результаты:", OUTPUT_DIR)

,parameter,value
0,grid_resolution_A,0.2
1,grid_search_cutoff_A,5.0
2,grid_symprec_A,0.1
3,grid_minima_window,3
4,grid_relative_max_value_A,None
5,grid_r_min_A,1.0
6,grid_empty_framework,False
7,grid_verbose,False
8,probe_element,'Li'
9,probe_oxidation_state,1


Python: /Users/irinabrodyagina/Desktop/electrolyties/elec_venv/bin/python
Результаты: /Users/irinabrodyagina/Desktop/РНФ_поиск_пор/elec_voids_results/notebook_tests_20261024_184054_101492


## Входные структуры

Все CIF из `kobernik_voids/cifs` и две структуры с искусственной Li-вакансией. Входные файлы не изменяются.

In [2]:
TEST_FOLDER = PROJECT_ROOT / "kobernik_voids/cifs"
VACANCY_FILES = [
    PROJECT_ROOT / "cif_tests_2/LiCl_one_Li_vacancy.cif",
    PROJECT_ROOT / "cif_tests_2/Li3PO4_one_Li_vacancy.cif",
]
INPUTS = sorted(p for p in TEST_FOLDER.iterdir() if p.is_file() and p.suffix.lower() == ".cif") + VACANCY_FILES
assert INPUTS and all(p.is_file() for p in INPUTS)
assert len({p.stem for p in INPUTS}) == len(INPUTS)
input_table = pd.DataFrame({
    "structure": [p.stem for p in INPUTS],
    "file": [str(p.relative_to(PROJECT_ROOT)) for p in INPUTS],
    "sha256": [hashlib.sha256(p.read_bytes()).hexdigest() for p in INPUTS],
})
input_table.to_csv(OUTPUT_DIR / "inputs.csv", index=False)
display(input_table[["structure", "file"]])

,structure,file
0,LATP_sc_integer152_stoichiometric,kobernik_voids/cifs/LATP_sc_integer152_stoichi...
1,LLZO_cub_ga,kobernik_voids/cifs/LLZO_cub_ga.cif
2,Li10Ge(PS6)2,kobernik_voids/cifs/Li10Ge(PS6)2.cif
3,Li3PO4,kobernik_voids/cifs/Li3PO4.cif
4,LiCl,kobernik_voids/cifs/LiCl.cif
5,LiCl_one_Li_vacancy,cif_tests_2/LiCl_one_Li_vacancy.cif
6,Li3PO4_one_Li_vacancy,cif_tests_2/Li3PO4_one_Li_vacancy.cif


## Источники и отсутствие консольных аргументов

Вычислительные блоки взяты из `test_easy.ipynb` и `single_structure_voronoi.ipynb`. Для доступности всех настроек параметризованы сигнатуры и значения внутри вызовов. Поэтому целиком сравниваются только функции, которые остались дословными копиями; остальные изменения описаны в файлах происхождения и PARAMETERS.md. Исходные notebook и исходный `void_search.py` не перезаписываются.

In [3]:
package_dir = PROJECT_ROOT / "elec_voids_search"
original_paths = [PROJECT_ROOT / "kobernik_voids/src/test_easy.ipynb",
                  PROJECT_ROOT / "kobernik_voids/src/void_search.py",
                  PROJECT_ROOT / "single_structure_voronoi.ipynb"]
original_hashes = {str(path): hashlib.sha256(path.read_bytes()).hexdigest() for path in original_paths}
copied = json.loads((package_dir / "COPIED_FUNCTIONS.json").read_text())
for entry in copied:
    hashes = []
    for key in ["source", "destination"]:
        text = (PROJECT_ROOT / entry[key]).read_text()
        node = next(n for n in ast.parse(text).body if isinstance(n, ast.FunctionDef) and n.name == entry["function"])
        block = "".join(text.splitlines(keepends=True)[node.lineno-1:node.end_lineno]).rstrip(chr(10))
        hashes.append(hashlib.sha256(block.encode()).hexdigest())
    assert hashes[0] == hashes[1] == entry["function_source_sha256"], entry["function"]
main_source = (package_dir / "run_search.py").read_text()
assert not any(token in main_source for token in ["argparse", "parse_args", "sys.argv"])
print("Дословные функции соответствуют источникам; все остальные параметры заданы через SearchConfig.")
geometry_sources = json.loads((package_dir / "GEOMETRY_SOURCE.json").read_text())
for entry in geometry_sources["module_copies"]:
    source_bytes = (PROJECT_ROOT / entry["source"]).read_bytes()
    destination_bytes = (PROJECT_ROOT / entry["destination"]).read_bytes()
    assert source_bytes == destination_bytes
    assert hashlib.sha256(source_bytes).hexdigest() == entry["sha256"]
print("shannon_radii.py и site_atomic_data.py совпадают с оригиналами побайтно.")


Дословные функции соответствуют источникам; все остальные параметры заданы через SearchConfig.
shannon_radii.py и site_atomic_data.py совпадают с оригиналами побайтно.


## Запуск всех структур и сохранение этапов

`geometry` содержит прошедшие кандидаты; `geometry_indeterminate` — случаи с недостающими данными; `geometry_rejected` — геометрические перекрытия. `geometry_retained` объединяет прошедшие и неопределённые для дальнейшего анализа, но не называет неопределённые прошедшими.

`geometry_evaluated` содержит все симметрийные группы после выбора лучшей исходной точки и ограниченного локального уточнения. ID относится к исходному представителю группы; фактическая начальная точка указана в `geometry_seed_candidate_id`. Уточнённые координаты могут отличаться от исходных. Исходные файлы и ранние таблицы сохраняются.

Ошибка одной структуры записывается в `failures.csv`; остальные структуры продолжают рассчитываться.


In [4]:
results, summaries, failures = {}, [], []
for number, path in enumerate(INPUTS, 1):
    print(f"[{number}/{len(INPUTS)}] {path.name}", flush=True)
    started = perf_counter()
    try:
        with (LOG_DIR / f"{path.stem}.log").open("w", buffering=1) as log, redirect_stdout(log), redirect_stderr(log):
            result = run_search(path, OUTPUT_DIR / path.stem, config=SETTINGS)
    except Exception as exc:
        failures.append({"structure": path.stem, "error": repr(exc)})
        pd.DataFrame(failures).to_csv(OUTPUT_DIR / "failures.csv", index=False)
        print("Ошибка:", repr(exc), flush=True)
        continue
    results[path.stem] = result
    summary = result["summary"].copy()
    summary["elapsed_seconds"] = perf_counter() - started
    summaries.append(summary)
    pd.concat(summaries, ignore_index=True).to_csv(OUTPUT_DIR / "all_structures_summary.csv", index=False)
    print("После симметрии:", len(result["nonequivalent"]),
          "; геометрия:", len(result["geometry"]),
          "; неопределённо:", len(result["geometry_indeterminate"]), flush=True)
all_summary = pd.concat(summaries, ignore_index=True) if summaries else pd.DataFrame()
display(all_summary)
display(pd.DataFrame(failures, columns=["structure", "error"]))


[1/7] LATP_sc_integer152_stoichiometric.cif


После симметрии: 460 ; геометрия: 197 ; неопределённо: 179


[2/7] LLZO_cub_ga.cif


После симметрии: 1404 ; геометрия: 0 ; неопределённо: 1235


[3/7] Li10Ge(PS6)2.cif


...

После симметрии: 16 ; геометрия: 0 ; неопределённо: 16


[7/7] Li3PO4_one_Li_vacancy.cif


После симметрии: 60 ; геометрия: 13 ; неопределённо: 31


,structure_file,formula,space_group,n_grid_candidates,n_voronoi_raw_vertices,n_voronoi_candidates,n_combined_candidates,n_merged_candidates,n_symmetry_nonequivalent_candidates,n_geometry_passed,...,n_full_radius_passed,probe_element,probe_oxidation_state,geometry_minimum_fit_fraction,grid_resolution_A,merge_distance_A,merge_representative,symprec_A,symmetry_tolerance_A,elapsed_seconds
0,LATP_sc_integer152_stoichiometric.cif,Li5Ti7Al(PO4)12,Pc,278,936,936,1214,916,460,197,...,197,Li,1,1.0,0.2,0.1,voronoi,0.1,0.1,94.883684
1,LLZO_cub_ga.cif,Li25La12Zr8GaO48,P1,519,1271,1271,1790,1404,1404,0,...,0,Li,1,1.0,0.2,0.1,voronoi,0.1,0.1,216.339028
2,Li10Ge(PS6)2.cif,Li10Ge(PS6)2,P4_2mc,76,330,330,406,344,55,0,...,0,Li,1,1.0,0.2,0.1,voronoi,0.1,0.1,18.441804
3,Li3PO4.cif,Li3PO4,Pmn2_1,24,116,116,140,90,28,5,...,5,Li,1,1.0,0.2,0.1,voronoi,0.1,0.1,11.628341
4,LiCl.cif,LiCl,P6_3mc,16,10,10,26,18,5,0,...,0,Li,1,1.0,0.2,0.1,voronoi,0.1,0.1,1.461100
5,LiCl_one_Li_vacancy.cif,LiCl2,P3m1,19,11,11,30,27,16,0,...,0,Li,1,1.0,0.2,0.1,voronoi,0.1,0.1,2.205313
6,Li3PO4_one_Li_vacancy.cif,Li5(PO4)2,Pm,26,104,104,130,94,60,13,...,13,Li,1,1.0,0.2,0.1,voronoi,0.1,0.1,17.092555


,structure,error


## Проверки порядка этапов, координат и происхождения

Проверяются сохранение обеих исходных выборок до группировки, полное покрытие ID кластерами, принадлежность каждого представителя исходному списку и запись действующих настроек. Координаты не должны усредняться. Дополнительная колонка показывает размах кластера относительно его представителя.

In [5]:
table_checks = []
for name, result in results.items():
    grid, voronoi = result["grid"], result["voronoi"]
    combined, merged = result["combined"], result["merged"]
    selected, clusters = result["nonequivalent"], result["clusters"]
    assert len(combined) == len(grid) + len(voronoi)
    assert len(voronoi) == result["summary"].iloc[0].n_voronoi_raw_vertices
    assert combined.candidate_id.is_unique
    assert set(merged.candidate_id) == set(combined.loc[combined.is_merge_representative, "candidate_id"])
    assert set(selected.candidate_id) <= set(merged.candidate_id)
    assert set(selected.candidate_id) == set(merged.loc[merged.is_symmetry_representative, "candidate_id"])
    all_members = [candidate for members in clusters.member_candidate_ids for candidate in members]
    assert sorted(all_members) == sorted(combined.candidate_id.tolist())
    assert len(all_members) == len(combined)
    for method, original in [("grid", grid), ("voronoi", voronoi)]:
        part = combined[combined.source_method == method]
        assert part.source_candidate_id.tolist() == original.candidate_id.tolist()
        np.testing.assert_allclose(part[["frac_x", "frac_y", "frac_z"]], original[["frac_x", "frac_y", "frac_z"]], atol=0, rtol=0)
    original_rows = combined.set_index("candidate_id")
    np.testing.assert_allclose(merged[["frac_x", "frac_y", "frac_z"]],
                               original_rows.loc[merged.candidate_id, ["frac_x", "frac_y", "frac_z"]], atol=0, rtol=0)
    for stage in STAGES:
        assert len(pd.read_csv(OUTPUT_DIR / name / f"{stage}_candidates.csv")) == len(result[stage])
    evaluated = result["geometry_evaluated"]
    assert len(evaluated) == len(selected)
    assert len(result["geometry"]) + len(result["geometry_rejected"]) + len(result["geometry_indeterminate"]) == len(selected)
    assert set(evaluated.candidate_id) == set(selected.candidate_id)
    assert set(result["geometry_seed_audit"].candidate_id) == set(combined.candidate_id)
    assert (result["geometry"].geometry_status == "passed").all()
    if len(evaluated):
        host = Structure.from_file(next(p for p in INPUTS if p.stem == name))
        np.testing.assert_allclose(evaluated[["cart_x_A", "cart_y_A", "cart_z_A"]],
                                   evaluated[["frac_x", "frac_y", "frac_z"]].to_numpy() @ host.lattice.matrix,
                                   atol=1e-9, rtol=0)
    metadata = json.loads((OUTPUT_DIR / name / "metadata.json").read_text())
    assert metadata["parameters"]["settings"] == SETTINGS.to_dict() == result["settings"]
    table_checks.append({"structure": name, "passed": True,
                         "max_cluster_span_A": float(clusters.max_distance_to_representative_A.max()) if len(clusters) else 0})
display(pd.DataFrame(table_checks))

/Users/irinabrodyagina/Desktop/electrolyties/elec_venv/lib/python3.12/site-packages/pymatgen/core/structure.py:3117: UserWarning: Issues encountered while parsing CIF: 8 fractional coordinates rounded to ideal values to avoid issues with finite precision.
  struct = parser.parse_structures(primitive=primitive)[0]
/Users/irinabrodyagina/Desktop/electrolyties/elec_venv/lib/python3.12/site-packages/pymatgen/core/structure.py:3117: UserWarning: Issues encountered while parsing CIF: 6 fractional coordinates rounded to ideal values to avoid issues with finite precision.
  struct = parser.parse_structures(primitive=primitive)[0]


,structure,passed,max_cluster_span_A
0,LATP_sc_integer152_stoichiometric,True,0.183074
1,LLZO_cub_ga,True,0.177219
2,Li10Ge(PS6)2,True,0.163350
3,Li3PO4,True,0.131663
4,LiCl,True,0.087439
5,LiCl_one_Li_vacancy,True,0.099494
6,Li3PO4_one_Li_vacancy,True,0.122329


## Контроль известных вакансий

Эталон определяется из исходной и дефектной структуры. Порог 0,25 Å оценивает восстановление известного Li; он не участвует в группировке или поиске. Новые значения допусков не считаются валидированными до выполнения этих проверок.

In [6]:
def get_removed_li_position(pristine, defect):
    np.testing.assert_allclose(pristine.lattice.matrix, defect.lattice.matrix, atol=1e-7)
    assert len(pristine) == len(defect) + 1
    distance = pristine.lattice.get_all_distances(pristine.frac_coords, defect.frac_coords)
    missing, owners = [], []
    for i, site in enumerate(pristine):
        same_element = np.array([other.specie.symbol == site.specie.symbol for other in defect])
        matches = np.flatnonzero(same_element & (distance[i] < 1e-6))
        if len(matches):
            assert len(matches) == 1
            owners.append(int(matches[0]))
        else:
            missing.append(i)
    assert len(missing) == 1 and pristine[missing[0]].specie.symbol == "Li"
    assert sorted(owners) == list(range(len(defect)))
    return pristine[missing[0]].frac_coords

vacancy_rows = []
for material in (["LiCl", "Li3PO4"] if SETTINGS.probe_element == "Li" and SETTINGS.probe_oxidation_state == 1 else []):
    pristine = Structure.from_file(PROJECT_ROOT / f"cif_tests_2/{material}.cif")
    name = f"{material}_one_Li_vacancy"
    defect = Structure.from_file(PROJECT_ROOT / f"cif_tests_2/{name}.cif")
    if name not in results:
        continue
    target = get_removed_li_position(pristine, defect)
    operations = SpacegroupAnalyzer(defect, symprec=SETTINGS.symmetry_symprec_A,
                                   angle_tolerance=SETTINGS.symmetry_angle_tolerance_deg).get_symmetry_operations()
    target_orbit = np.array([operation.operate(target) % 1 for operation in operations])
    for stage in STAGES:
        table = results[name][stage]
        nearest = float(defect.lattice.get_all_distances([target], table[["frac_x", "frac_y", "frac_z"]]).min()) if len(table) else float("inf")
        orbit_nearest = float(defect.lattice.get_all_distances(target_orbit, table[["frac_x", "frac_y", "frac_z"]]).min()) if len(table) else float("inf")
        vacancy_rows.append({"structure": name, "stage": stage,
                             "nearest_vacancy_A": nearest, "nearest_symmetry_orbit_A": orbit_nearest,
                             "recovered": orbit_nearest < VACANCY_TOLERANCE_A})
vacancy_recovery = pd.DataFrame(vacancy_rows, columns=["structure", "stage", "nearest_vacancy_A", "nearest_symmetry_orbit_A", "recovered"])
vacancy_recovery.to_csv(OUTPUT_DIR / "vacancy_recovery.csv", index=False)
display(vacancy_recovery)
# Не прерываем статистику при потере вакансии: это результат сравнения.
# Отброшенные/неопределённые таблицы диагностические, не этапы успешного отбора.
recovery_stages = ["grid", "voronoi", "combined", "merged", "nonequivalent", "geometry", "geometry_retained"]
display(vacancy_recovery[vacancy_recovery.stage.isin(recovery_stages)].pivot(index="structure", columns="stage", values="recovered"))

/Users/irinabrodyagina/Desktop/electrolyties/elec_venv/lib/python3.12/site-packages/pymatgen/core/structure.py:3117: UserWarning: Issues encountered while parsing CIF: 8 fractional coordinates rounded to ideal values to avoid issues with finite precision.
  struct = parser.parse_structures(primitive=primitive)[0]
/Users/irinabrodyagina/Desktop/electrolyties/elec_venv/lib/python3.12/site-packages/pymatgen/core/structure.py:3117: UserWarning: Issues encountered while parsing CIF: 6 fractional coordinates rounded to ideal values to avoid issues with finite precision.
  struct = parser.parse_structures(primitive=primitive)[0]


,structure,stage,nearest_vacancy_A,nearest_symmetry_orbit_A,recovered
0,LiCl_one_Li_vacancy,grid,1.524638e-01,1.524638e-01,True
1,LiCl_one_Li_vacancy,voronoi,1.655591e-04,1.655591e-04,True
2,LiCl_one_Li_vacancy,combined,1.655591e-04,1.655591e-04,True
3,LiCl_one_Li_vacancy,merged,1.655591e-04,1.655591e-04,True
4,LiCl_one_Li_vacancy,nonequivalent,1.655591e-04,1.655591e-04,True
5,LiCl_one_Li_vacancy,geometry_evaluated,2.187620e-07,2.187620e-07,True
6,LiCl_one_Li_vacancy,geometry,inf,inf,False
7,LiCl_one_Li_vacancy,geometry_indeterminate,2.187620e-07,2.187620e-07,True
8,LiCl_one_Li_vacancy,geometry_rejected,inf,inf,False
9,LiCl_one_Li_vacancy,geometry_retained,2.187620e-07,2.187620e-07,True


stage,combined,geometry,geometry_retained,grid,merged,nonequivalent,voronoi
structure,,,,,,,
Li3PO4_one_Li_vacancy,True,True,True,True,True,True,True
LiCl_one_Li_vacancy,True,False,True,True,True,True,True


## Общие координаты и проверка полного размера иона

Следующие таблицы объединяют все успешно рассчитанные структуры. Для каждого этапа сохраняются координаты, исходные ID и результаты геометрии. `full_radius_status` всегда проверяет полный найденный радиус пробника. Основной критерий использует geometry_minimum_fit_fraction (дефолт 1.0); при дефолте критерии совпадают. При явном включении fallback источник радиуса записывается отдельно.

Контроли занятых сайтов выбранного пробного элемента используют удаление центрального сайта из окружения, без оптимизации. Берётся радиус самого занятого сайта с его исходным окружением; проверка не является независимым поиском неизвестного междоузлия. Координаты искусственных вакансий не передаются в поиск. Контроли сохраняются только для выбранного элемента и заряда.


In [7]:
coordinate_tables, counts, model_rows, control_tables, radius_tables = [], [], [], [], []
for name, result in results.items():
    for stage in STAGES:
        table = result[stage].copy()
        table.insert(0, "structure", name)
        table.insert(1, "stage", stage)
        coordinate_tables.append(table)
        counts.append({"structure": name, "stage": stage, "n_candidates": len(table)})
    evaluated = result["geometry_evaluated"]
    for primary in ["passed", "indeterminate", "rejected"]:
        for strict in ["passed", "indeterminate", "rejected"]:
            model_rows.append({"structure": name, "geometry_status": primary,
                               "full_radius_status": strict,
                               "count": int(((evaluated.geometry_status == primary) &
                                             (evaluated.full_radius_status == strict)).sum())})
    for key, target in [("occupied_probe_controls", control_tables), ("site_radii", radius_tables)]:
        table = result[key].copy()
        table.insert(0, "structure", name)
        target.append(table)
counts = pd.DataFrame(counts, columns=["structure", "stage", "n_candidates"])
counts.to_csv(OUTPUT_DIR / "stage_counts.csv", index=False)
if len(counts):
    display(counts.pivot(index="structure", columns="stage", values="n_candidates").reindex(columns=STAGES))
for filename, tables in [("all_stage_coordinates", coordinate_tables),
                         ("all_occupied_probe_controls", control_tables), ("all_site_radii", radius_tables)]:
    table = pd.concat(tables, ignore_index=True) if tables else pd.DataFrame()
    table.to_csv(OUTPUT_DIR / f"{filename}.csv", index=False)
    (OUTPUT_DIR / f"{filename}.json").write_text(table.to_json(orient="records", indent=2, double_precision=15))
    if filename != "all_stage_coordinates":
        display(table)
model_comparison = pd.DataFrame(model_rows, columns=["structure", "geometry_status", "full_radius_status", "count"])
model_comparison.to_csv(OUTPUT_DIR / "geometry_fit_fraction_comparison.csv", index=False)
display(model_comparison[model_comparison["count"] > 0])
# Координаты для просмотра (полные данные сохранены в CSV/JSON и CIF с X):
for name, result in results.items():
    print(name)
    display(result["geometry"][["candidate_id", "frac_x", "frac_y", "frac_z",
                                "probe_element", "probe_oxidation_state", "probe_coordination",
                                "probe_radius_A", "probe_radius_source", "free_radius_A",
                                "geometry_margin_A", "full_radius_margin_A"]])


stage,grid,voronoi,combined,merged,nonequivalent,geometry_evaluated,geometry,geometry_indeterminate,geometry_rejected,geometry_retained
structure,,,,,,,,,,
LATP_sc_integer152_stoichiometric,278,936,1214,916,460,460,197,179,84,376
LLZO_cub_ga,519,1271,1790,1404,1404,1404,0,1235,169,1235
Li10Ge(PS6)2,76,330,406,344,55,55,0,52,3,52
Li3PO4,24,116,140,90,28,28,5,19,4,24
Li3PO4_one_Li_vacancy,26,104,130,94,60,60,13,31,16,44
LiCl,16,10,26,18,5,5,0,5,0,5
LiCl_one_Li_vacancy,19,11,30,27,16,16,0,16,0,16


,structure,site_index,probe_element,probe_oxidation_state,probe_coordination,probe_radius_A,free_radius_A,geometry_margin_A,geometry_status,frac_x,frac_y,frac_z
0,LATP_sc_integer152_stoichiometric,0,Li,1,6,0.76,0.866783,0.106783,passed,0.000000,0.000000,0.750000
1,LATP_sc_integer152_stoichiometric,1,Li,1,6,0.76,0.866783,0.106783,passed,0.500000,0.500000,0.500000
2,LATP_sc_integer152_stoichiometric,2,Li,1,6,0.76,0.866783,0.106783,passed,0.500000,0.500000,0.750000
3,LATP_sc_integer152_stoichiometric,3,Li,1,6,0.76,0.866783,0.106783,passed,0.000000,0.000000,0.250000
4,LATP_sc_integer152_stoichiometric,4,Li,1,6,0.76,0.866783,0.106783,passed,0.000000,0.000000,0.500000
...,...,...,...,...,...,...,...,...,...,...,...,...
89,Li3PO4_one_Li_vacancy,0,Li,1,4,0.59,0.562467,-0.027533,rejected,0.009716,0.157865,0.500000
90,Li3PO4_one_Li_vacancy,1,Li,1,4,0.59,0.536445,-0.053555,rejected,0.509058,0.327715,0.251526
91,Li3PO4_one_Li_vacancy,2,Li,1,4,0.59,0.516445,-0.073555,rejected,0.009058,0.672285,0.248474
92,Li3PO4_one_Li_vacancy,3,Li,1,4,0.59,0.516445,-0.073555,rejected,0.009058,0.672285,0.751526


,structure,site_index,element,coordination,oxidation_state,atomic_radius_A,ionic_radius_A,shannon_radius_A,radius_A,radius_source,radius_issue,radius_warning
0,LATP_sc_integer152_stoichiometric,0,Li,6,1.0,1.45,0.90,0.76,0.76,shannon,,
1,LATP_sc_integer152_stoichiometric,1,Li,6,1.0,1.45,0.90,0.76,0.76,shannon,,
2,LATP_sc_integer152_stoichiometric,2,Li,6,1.0,1.45,0.90,0.76,0.76,shannon,,
3,LATP_sc_integer152_stoichiometric,3,Li,6,1.0,1.45,0.90,0.76,0.76,shannon,,
4,LATP_sc_integer152_stoichiometric,4,Li,6,1.0,1.45,0.90,0.76,0.76,shannon,,
...,...,...,...,...,...,...,...,...,...,...,...,...
417,Li3PO4_one_Li_vacancy,10,O,3,-2.0,0.60,1.26,1.36,1.36,shannon,,
418,Li3PO4_one_Li_vacancy,11,O,3,-2.0,0.60,1.26,1.36,1.36,shannon,,
419,Li3PO4_one_Li_vacancy,12,O,4,-2.0,0.60,1.26,1.38,1.38,shannon,,
420,Li3PO4_one_Li_vacancy,13,O,3,-2.0,0.60,1.26,1.36,1.36,shannon,,


,structure,geometry_status,full_radius_status,count
0,LATP_sc_integer152_stoichiometric,passed,passed,197
4,LATP_sc_integer152_stoichiometric,indeterminate,indeterminate,179
8,LATP_sc_integer152_stoichiometric,rejected,rejected,84
13,LLZO_cub_ga,indeterminate,indeterminate,1235
17,LLZO_cub_ga,rejected,rejected,169
22,Li10Ge(PS6)2,indeterminate,indeterminate,52
26,Li10Ge(PS6)2,rejected,rejected,3
27,Li3PO4,passed,passed,5
31,Li3PO4,indeterminate,indeterminate,19
35,Li3PO4,rejected,rejected,4


LATP_sc_integer152_stoichiometric


...

,candidate_id,frac_x,frac_y,frac_z,probe_element,probe_oxidation_state,probe_coordination,probe_radius_A,probe_radius_source,free_radius_A,geometry_margin_A,full_radius_margin_A


LiCl_one_Li_vacancy


,candidate_id,frac_x,frac_y,frac_z,probe_element,probe_oxidation_state,probe_coordination,probe_radius_A,probe_radius_source,free_radius_A,geometry_margin_A,full_radius_margin_A


Li3PO4_one_Li_vacancy


,candidate_id,frac_x,frac_y,frac_z,probe_element,probe_oxidation_state,probe_coordination,probe_radius_A,probe_radius_source,free_radius_A,geometry_margin_A,full_radius_margin_A
2,3,0.327326,0.430343,5.000000e-01,Li,1,6,0.76,shannon,0.764761,0.004761,0.004761
5,105,0.377193,0.557238,1.000000e+00,Li,1,6,0.76,shannon,0.831219,0.071219,0.071219
9,13,0.509566,0.834033,1.000000e+00,Li,1,4,0.59,shannon,0.621002,0.031002,0.031002
12,111,0.822211,0.569024,3.373198e-15,Li,1,6,0.76,shannon,0.773690,0.013690,0.013690
15,108,0.874505,0.448383,5.000000e-01,Li,1,6,0.76,shannon,0.826947,0.066947,0.066947
28,41,0.368686,0.563170,9.875853e-01,Li,1,6,0.76,shannon,0.783293,0.023293,0.023293
32,46,0.829536,0.416217,4.339512e-01,Li,1,4,0.59,shannon,0.606127,0.016127,0.016127
34,48,0.377193,0.557238,1.470611e-13,Li,1,6,0.76,shannon,0.831219,0.071219,0.071219
38,63,0.327326,0.430343,5.000000e-01,Li,1,6,0.76,shannon,0.764761,0.004761,0.004761
46,82,0.325451,0.430575,5.000000e-01,Li,1,6,0.76,shannon,0.760678,0.000678,0.000678


In [8]:
for path, expected_hash in original_hashes.items():
    assert hashlib.sha256(Path(path).read_bytes()).hexdigest() == expected_hash
validation = {"structures_tested": len(results), "failures": failures, "table_checks": table_checks,
              "vacancy_checks": vacancy_recovery.to_dict("records"),
              "original_files_unchanged": True, "settings": SETTINGS.to_dict(),
              "stage_order": ["grid_and_voronoi", "combined", "merged", "nonequivalent", "geometry"]}
(OUTPUT_DIR / "validation.json").write_text(json.dumps(validation, ensure_ascii=False, indent=2))
print("Структурные проверки таблиц завершены. Восстановление вакансий оцените по vacancy_recovery.csv. Результаты:", OUTPUT_DIR)

Структурные проверки таблиц завершены. Восстановление вакансий оцените по vacancy_recovery.csv. Результаты: /Users/irinabrodyagina/Desktop/РНФ_поиск_пор/elec_voids_results/notebook_tests_20261024_184054_101492


## Заключительная проверка после уточнения

Эта дополнительная ячейка выполнена отдельно на сохранённых результатах, без повторения поиска. Проверены сохранение всех прошедших ID и размерные неравенства; близкие и симметрийно эквивалентные финальные позиции объединены.

In [9]:
from pathlib import Path
import json
import numpy as np
import pandas as pd
from pymatgen.core import Structure
from pymatgen.symmetry.analyzer import SpacegroupAnalyzer
from elec_voids_search.config import SearchConfig
from elec_voids_search.final_selection import final_unique_candidates
from elec_voids_search.exports import export_candidates

# This cell can also run alone on the saved completed tables.
if 'PROJECT_ROOT' not in globals():
    PROJECT_ROOT = Path('/Users/irinabrodyagina/Desktop/РНФ_поиск_пор')
if 'OUTPUT_DIR' not in globals():
    OUTPUT_DIR = PROJECT_ROOT / 'elec_voids_results/notebook_tests_20261024_184054_101492'
settings = SearchConfig(**json.loads((OUTPUT_DIR / 'settings.json').read_text()))
inputs = pd.read_csv(OUTPUT_DIR / 'inputs.csv')
post_summary = pd.read_csv(OUTPUT_DIR / 'all_structures_summary.csv')
post_checks = []
for _, item in inputs.iterrows():
    folder = OUTPUT_DIR / item.structure
    host = Structure.from_file(PROJECT_ROOT / item.file)
    passed = pd.read_csv(folder / 'geometry_candidates.csv')
    analyzer = SpacegroupAnalyzer(host, symprec=settings.symmetry_symprec_A,
                                  angle_tolerance=settings.symmetry_angle_tolerance_deg)
    unique = final_unique_candidates(host, passed, analyzer, settings.symmetry_tolerance_A)
    members = [candidate for group in unique.final_member_candidate_ids for candidate in group]
    assert sorted(members) == sorted(passed.candidate_id.tolist())
    assert len(members) == len(set(members))
    assert (unique.geometry_status == 'passed').all()
    sites = pd.read_csv(folder / 'site_radii.csv')
    if len(unique):
        clearance = (host.lattice.get_all_distances(unique[['frac_x', 'frac_y', 'frac_z']],
                                                    host.frac_coords) - sites.radius_A.to_numpy()).min(axis=1)
        np.testing.assert_allclose(clearance, unique.free_radius_A, atol=1e-8, rtol=0)
        assert np.all(clearance + 1e-7 >= unique.probe_radius_A.to_numpy())
    export_candidates(host, unique, folder, 'geometry_unique',
                      {'settings': settings.to_dict(), 'stage': 'Final symmetry check after refinement'})
    post_summary.loc[post_summary.structure_file == Path(item.file).name, 'n_geometry_unique'] = len(unique)
    post_checks.append({'structure': item.structure, 'passed_before': len(passed),
                        'passed_unique': len(unique), 'members_preserved': True,
                        'full_radius_rechecked': True})
post_summary['n_geometry_unique'] = post_summary.n_geometry_unique.astype(int)
post_summary.to_csv(OUTPUT_DIR / 'all_structures_summary.csv', index=False)
(OUTPUT_DIR / 'final_selection_validation.json').write_text(json.dumps(post_checks, indent=2))
print(pd.DataFrame(post_checks).to_string(index=False))


                        structure  passed_before  passed_unique  members_preserved  full_radius_rechecked
LATP_sc_integer152_stoichiometric            197            108               True                   True
                      LLZO_cub_ga              0              0               True                   True
                     Li10Ge(PS6)2              0              0               True                   True
                           Li3PO4              5              2               True                   True
                             LiCl              0              0               True                   True
              LiCl_one_Li_vacancy              0              0               True                   True
            Li3PO4_one_Li_vacancy             13              6               True                   True
